In [15]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fsolve, differential_evolution

In [16]:
def m_func(value,Delta, Log_space=False):
    if Log_space:
        return np.ceil(np.exp(value*Delta))
    else:
        return np.ceil(np.power(value,Delta))


Requirement 1, lower bound on $\delta$, depending on $\epsilon$

In [17]:
def delta_bound(Epsilon):
    return 1/(1-Epsilon)*3/4


Requirement 2 on the value of $\log_2n$, the largest $\delta$ is, the smaller n becomes,
                                 the further $\epsilon$ is from its optimum, the largest n is

Optimum value of epsilon is $1/2 - \frac{3}{8\delta}$

In [18]:
def optimal_epsilon(Delta):
    return 1/2 - 3/(8*Delta)

def req_on_log2n(Delta,Epsilon):
    return 1/(2*Delta*Epsilon*(1-Epsilon) - 3*Epsilon/2)


Bound on $\mathbb{P}[D_R$ does not hold $]$

In [19]:
def bound_on_Dx(m1):
    return m1*np.exp(-np.sqrt(m1)/(3*np.log(m1)**2))

# for m in np.logspace(1,15,1000):
#     if bound_on_Dx(m) < 1e-18:
#         print(f"m = {m:.4e}")
#         for n in np.logspace(1,100,1000):
#             if m_func(n,delta) >= m:
#                 print(f"n = {n:.4e}")
#                 break
#         break

Bound on $\mathbb{P}[F_R$ does not hold $]$

In [20]:
def bound_on_Fx(value,Delta,Log_space=False):
    if Log_space:
        M1 = m_func(value,Delta, Log_space)
        return M1*np.exp(-np.exp(value*(1-Delta))/4)
    else:
        return m_func(value,Delta)*np.exp(-np.power(value,1-Delta)/4)

# for n in np.logspace(1,100,1000):
#     result = bound_on_Fx(n,delta)
#     if result < 1e-3:
#         print(f"Value of n = {n:.1e}, result is {result:2e}")
#         break

Bound on $P[T_R$ does not hold$]$, given by $\frac{1}{m}$

In [21]:
def bound_on_Tx(m1,K):
    return np.power(m1,-K/6+2)

# for n in np.logspace(1,300,1000):
#
#     if 1/m_func(n,delta)< 1e-12:
#         print(f"Value of n = {n:.1e}")
#         break

Bound on $\mathbb{P}[\mathcal{E}(\epsilon)^c]$

In [22]:
def bound_on_Eepsilon(value,Delta,Epsilon,Log_space=True):
    c0 = 1+ 1e-30
    beta1 = 3*Epsilon - 4*Delta*Epsilon + 4*Delta*Epsilon**2 + np.log(c0)*2*Epsilon**2
    beta2 = 3-4*Delta+4*Delta*Epsilon
    beta3 = 2/np.log2(np.exp(1))
    if Log_space:
        return np.exp(value*(beta1*np.exp(value)+beta2 ) + 2/np.log2(beta3))

From here, calculations for Lemma 4.4, first one is bound on $\mathbb{P}[\mathbb{H}$ doesnt hold$]$

In [23]:
def bound_on_H(value,Delta,Gamma,Constant1,Log_space=False):
    term = Gamma*Constant1
    if Log_space: #if in log space, returns only the exponent
        M1 = m_func(value,Delta,Log_space)
        return -term*np.exp(2*value)/M1
    else:
        m1 = m_func(value,Delta,Log_space)
        return np.exp(-term*np.power(value,2)/m1)


Bound on P[there is no $\{e,f\} \in M$ st no triangle in $H_R$ contains $e$ or $f | H_R = H_R^*] := A$

The bound on A depends on a parameter $\zeta$, where it satisfies $(1-\frac{1}{\log^2(m)})^{80\log(m)} \geq \zeta$

I do computations in log space, to get an estimate of n, ($L = \log n$)

In [24]:
def H_condition(value,Delta,Gamma,Constant1,threshold=-1e1):#constant 1 is the value that arises from lemma 7.2
    Alpha1 = Gamma*Constant1
    M1 = m_func(value,Delta,Log_space=True)
    return M1*value/Alpha1 - np.exp(value)< threshold

def A_condition(value,Delta,Gamma,Zeta,Constant,threshold=-1e1):
    Alpha2 = Zeta*Gamma/Constant
    M1 = m_func(value,Delta,Log_space=True)
    return (M1*np.power(np.log(M1),6))*value -Alpha2*np.exp(value) <threshold

Computations for theorem 3.1

Requirements on n, $(1-\frac{1}{\log^2(m)})^{80\log(m)} \geq \zeta$, from lemma 4.4 and thm3.1

In [25]:
def thm31_req(value,Delta,Epsilon, bound,Log_space=True):
    if Log_space:
        M1 = m_func(value,Delta,Log_space=Log_space)
        return bound > 4/M1 + 16/np.power(np.log(M1),4) + Epsilon**2/(4*np.exp(value))

def lemma44_req(value,Zeta):
    return np.power(1-1/value**2,80*value) >= Zeta

In [26]:
def least_L_with_params(delta,epsilon,gamma,zeta,k):
    # Assuming epsilon and delta_bound are defined
    params = {
        "delta": delta,
        "zeta": zeta,
        "gamma": gamma,
        "k": k
    }

    # Define requirements as (Current Value, Boolean Condition, Description)
    requirements = {
        "Delta": (delta, 1 > delta > max(delta_bound(epsilon), 0.8), f"max({delta_bound(epsilon):.2f}, 0.8) < δ < 1"),
        "Zeta": (zeta, 0 < zeta < 1, "0 < ζ < 1"),
        "Gamma": (f"{gamma:.2e}", 0 < gamma < epsilon ** 4 / 4, f"0 < γ < {epsilon**4 / 4:.2e}"),
        "k": (k, k > 12, "k > 12")
    }

    # Check for failures
    failed = [name for name, (val, met, desc) in requirements.items() if not met]

    if failed:
        # print("═══ PARAMETER CHECK FAILED ═══")
        # for name, (val, met, desc) in requirements.items():
        #     status = "✓" if met else "✗"
        #     print(f"{status} {name:6} : {val:<8} | Required: {desc}")
        #
        # print(f"\nNote: epsilon can be at most: {1 - 3/(4*delta):.4f}")
        return 1e12
        # raise ValueError(f"Constraint violation in: {', '.join(failed)}")

    # print("✓ All parameters satisfy requirements.")

    c = (4*k+1)*(2*k+1)
    C = 16*(c+1)
    for L in np.linspace(300,400,100):
        H_OK = H_condition(value=L, Delta=delta, Gamma=gamma, Constant1=3/416, threshold=1e-1)
        A_OK = A_condition(value=L, Delta=delta, Gamma=gamma, Zeta=zeta, Constant=C, threshold=1e-1)
        M = m_func(value=L,Delta=delta,Log_space=True)
        # print(f"L = {L}")
        # print(f"H bound {np.exp(L) *L-gamma/78*np.exp(2 * L)/M:.4e},{np.exp(L):.4e}")
        # print(f"A bound {-zeta*gamma/C *np.exp(2*L)/(M*np.power(np.log(M),6)) +L*np.exp(L):.4e},{np.exp(L):.4e}")
        # 2. Safety Check for big_element
        # If the exponent is huge, exp(exponent) is definitely > 1, so the condition fails.
        if not H_OK or not A_OK:
            continue # Skip to next L, this n is too small to satisfy the bound

        Fx_bound = bound_on_Fx(value=L,Delta=delta,Log_space=True)
        M = m_func(value=L,Delta=delta,Log_space=True)
        Dx_bound = bound_on_Dx(M)
        Tx_bound = bound_on_Tx(M,k)
        Ee_bound = bound_on_Eepsilon(value=L,Delta=delta,Epsilon=epsilon,Log_space=True)
        F_check = Fx_bound < 1e-1
        D_check = Dx_bound < 1e-1
        T_check = Tx_bound < 1e-1
        Ee_check = Ee_bound < 1e-1
        m = m_func(L,Delta=delta,Log_space=True)
        zeta_req = lemma44_req(np.log(m),zeta)
        req_thm31 = thm31_req(L,delta,epsilon,epsilon**4/4-gamma)
        req_log2n =  L/np.log(2) > req_on_log2n(delta,epsilon)
        cond = req_thm31 and req_log2n and zeta_req
        # print(req_thm31, req_log2n, zeta_req)
        if F_check and D_check and T_check and Ee_check and cond:
            print(f"logn is around {L:.4e} and so n is around {np.exp(L):.8e}")
            return L
    return 1e12

In [27]:
def GEMINI_least_L_with_params(params):
    # Differential Evolution passes parameters as a single list/array
    delta, epsilon, gamma, zeta, k = params

    # --- 1. Fast Constraint Check ---
    # We check these first to avoid expensive calculations
    if not (1 > delta > max(delta_bound(epsilon), 0.8)): return 1e12
    if not (0 < zeta < 1): return 1e12
    if not (0 < gamma < epsilon ** 4 / 4): return 1e12
    if not (k > 12): return 1e12

    # --- 2. Setup Constants ---
    c = (4*k + 1) * (2*k + 1)
    C = 16 * (c + 1)

    # --- 3. Search for L ---
    # Note: We return the FIRST L that works.
    # Since we want to minimize L, this is the 'least L' for these parameters.
    for L in np.linspace(320, 345, 30):
        H_OK = H_condition(value=L, Delta=delta, Gamma=gamma, Constant1=3/416, threshold=0.1)
        A_OK = A_condition(value=L, Delta=delta, Gamma=gamma, Zeta=zeta, Constant=C, threshold=0.1)

        if not (H_OK and A_OK):
            continue

        M = m_func(value=L, Delta=delta, Log_space=True)

        # Bounds checks
        F_check = bound_on_Fx(value=L, Delta=delta, Log_space=True) < 0.1
        D_check = bound_on_Dx(M) < 0.1
        T_check = bound_on_Tx(M, k) < 0.1
        Ee_check = bound_on_Eepsilon(value=L, Delta=delta, Epsilon=epsilon, Log_space=True) < 0.1

        # Logical Requirements
        zeta_req = lemma44_req(np.log(M), zeta)
        req_thm31 = thm31_req(L, delta, epsilon, (epsilon**4 / 4) - gamma)
        req_log2n = L / np.log(2) > req_on_log2n(delta, epsilon)

        if all([F_check, D_check, T_check, Ee_check, zeta_req, req_thm31, req_log2n]):
            return L # Found the smallest L for this param set

    return 1e12 # No L in 1-400 range satisfied the conditions

In [28]:
K=12.1
DELTA = 0.81
EPSILON= optimal_epsilon(DELTA)
GAMMA = EPSILON**4/10
ZETA = 0.725
least_L_with_params(delta=DELTA,epsilon=EPSILON,gamma=GAMMA,zeta=ZETA,k=K)




logn is around 3.4444e+02 and so n is around 3.89333294e+149


np.float64(344.44444444444446)

In [ ]:
# Define bounds for: [delta, epsilon, gamma, zeta, k]
bounds = [
    (0.8, 0.82),      # delta
    (0.01, 0.24),     # epsilon
    (1e-8, 0.001),    # gamma
    (0.01, 0.99),     # zeta
    (12.01, 15.0)     # k (increased upper bound slightly for search room)
]

result = differential_evolution(
    GEMINI_least_L_with_params,
    bounds,
    popsize=10,
    mutation=(0.5, 1),
    recombination=0.7,
    workers=-1  # Uses all CPU cores to run iterations in parallel!
)

if result.success:
    d, eps, g, z, k = result.x
    print(f"Optimal L found: {result.fun}")
    print(f"Params: δ={d:.4f}, ε={eps:.4f}, γ={g:.2e}, ζ={z:.4f}, k={k:.4f}")
else:
    print("Optimization failed:", result.message)

C:\Users\Laurent Costa\PyCharmMiscProject\.venv\Lib\site-packages\scipy\optimize\_differentialevolution.py:518: UserWarning: differential_evolution: the 'workers' keyword has overridden updating='immediate' to updating='deferred'
  with DifferentialEvolutionSolver(func, bounds, args=args,
